## Explore Dataset

In [1]:
import geopandas as gpd
import dotenv
import os
from addCSOs import calculate_risk_factors_cso

In [2]:
dotenv.load_dotenv()
RIVERS_URL = os.getenv("RIVERS_URL")

## Obtaining data

Keeping all the data in the EPSG 3857 coordinate referencing system (CRS) to work better with distances

### CSO Data

In [3]:
geo_JSON_url = "https://services2.arcgis.com/g6o32ZDQ33GpCIu3/arcgis/rest/services/Thames_Water_Storm_Overflow_Activity_(Production)_view/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson"
cso: gpd.GeoDataFrame = gpd.read_file(geo_JSON_url)

In [4]:
cso = cso.to_crs(epsg=3857)

In [ ]:
cso.head()

### River data

In [6]:
rivers = gpd.read_file(RIVERS_URL)
rivers = rivers.to_crs(epsg=3857)

In [7]:
rivers = rivers.clip(cso.total_bounds).explode()

## Use addCSOs.py

In [8]:
river_points, cso_with_locations = calculate_risk_factors_cso(
    cso,
    rivers,
    cso_df_id_col="Id",
    river_resolution=125,
    max_outfall_dist_from_river=700,
    max_risk_distance=3000,
    include_upstream=True,
    river_directed=False,
)

In [9]:
def simple_risk(csoInfo: list[tuple]):
    tot = 0
    for csoID, dist in csoInfo:
        tot += 3000 - abs(dist)
    return tot

In [10]:
river_points["Risk"] = river_points["csoInfo"].apply(simple_risk)

In [11]:
river_points = river_points[river_points["Risk"] > 0]

## Explore

In [ ]:
m = rivers.explore(
    color="lightgrey"
)
river_points.explore(
    m=m,
    column="Risk",
    cmap="viridis",
    vmax=15000,
)
cso_with_locations.explore(
    m=m,
    color="red",
    marker_kwds={"radius": 5}
)
cso_with_locations.set_geometry("outfallLocation").explore(
    m=m,
    color="pink",
    marker_kwds={"radius": 5}
)